# 05. Streaming: muitas métricas, guardrails e correção múltipla

**Setor:** streaming de vídeo. **Decisão:** lançar o novo algoritmo de
recomendação? O problema não é uma métrica, são **várias**: tempo assistido
(primária), número de sessões, taxa de conclusão, e um **guardrail** de buffering
(não pode piorar). Testar muitas métricas infla a chance de um falso-positivo, então
precisamos **corrigir para múltiplos testes**.

Um único experimento, uma única alocação, várias métricas. Fechamos com um
`ExperimentPipeline` e um `ExperimentReport`. Base teórica:
[IV. Inferência](../../../docs/pt-br/teoria/04-inferencia.md) (tópico 16) e
[V. Diagnósticos](../../../docs/pt-br/teoria/05-diagnosticos.md) (tópico 21).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "streaming_metrics.csv")
print(df.shape)
df.head()


## 1. Uma alocação, várias métricas

Randomizamos os usuários **uma vez** e reusamos a **mesma** alocação para todas
as métricas (é o mesmo experimento). Para cada métrica, construímos o resultado
observado e estimamos o efeito com `NeymanCI`.


In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI

design = CRD(p=0.5, seed=505)
base = design.randomize(df[["plan"]].copy())
t = base.data_[base.treatment_col_].to_numpy()

metrics = ["watch_time", "sessions", "completion", "buffering"]
results = {}
for m in metrics:
    data = base.data_.copy()
    data[m] = np.where(t == 1, df[f"{m}_y1"].to_numpy(), df[f"{m}_y0"].to_numpy())
    a = CRDAssignment(data=data, treatment_col=base.treatment_col_, design=design, seed=505)
    results[m] = NeymanCI(estimator=DifferenceInMeans(outcome_col=m)).fit(a).estimate()

{m: round(r.p_value, 4) for m, r in results.items()}


## 2. Correção para múltiplos testes (FWER, Holm)

Com quatro métricas, controlamos a taxa de erro por família com `Holm`. O
`ExperimentComparison` aplica a correção sobre a família e devolve uma tabela
pronta, além do forest plot.


In [ ]:
from skxperiments.pipeline import ExperimentComparison
from skxperiments.reporting import plot_forest

comparison = ExperimentComparison(correction="holm", alpha=0.05).run(results)
display_cols = ["experiment", "ate", "p_value", "p_value_corrected", "significant"]
print(comparison.table[display_cols].round(4).to_string(index=False))

ax = plot_forest(comparison)
ax.set_title("Effect by metric (Holm-corrected)")
ax.figure


## 3. Pipeline e relatório na métrica primária

Só o **tempo assistido** sobrevive à correção; sessões e conclusão não são
significativas, e o guardrail de **buffering não é sinalizado** (não houve
piora). Fechamos com o `ExperimentPipeline` na métrica primária, que roda o
`SRMTest` automaticamente, e geramos um `ExperimentReport` em HTML.


In [ ]:
from skxperiments.diagnostics import SRMTest
from skxperiments.pipeline import ExperimentPipeline

data = base.data_.copy()
data["watch_time"] = np.where(t == 1, df["watch_time_y1"].to_numpy(), df["watch_time_y0"].to_numpy())
primary = CRDAssignment(data=data, treatment_col=base.treatment_col_, design=design, seed=505)

pipeline_result = ExperimentPipeline(
    inference=NeymanCI(estimator=DifferenceInMeans(outcome_col="watch_time")),
    diagnostics=[SRMTest()],
).run(primary)
print(f"watch_time ATE={pipeline_result.results.ate:.3f}, "
      f"p={pipeline_result.results.p_value:.4f}, flagged={pipeline_result.flagged}")


In [ ]:
from IPython.display import HTML

from skxperiments.reporting import ExperimentReport

report = ExperimentReport(pipeline_result, title="Streaming recommender - watch time")
HTML(report.to_html())


## Decisão

O efeito verdadeiro em tempo assistido é `+3` minutos, e é o único real; as
demais métricas foram geradas sem efeito (ou efeito minúsculo). A correção de
Holm faz exatamente o trabalho certo: mantém a descoberta primária e não deixa o
ruído das outras métricas virar "vitória". Com o guardrail limpo, a recomendação
é **lançar**, monitorando buffering.

Isso fecha a trilha de use cases. Para os fundamentos por trás de cada peça, veja
a [série de teoria](../../../docs/pt-br/teoria/01-fundamentos.md).
